# R1 Star Schema

In [0]:
use catalog claims;
create schema if not exists analytics;

In [0]:
CREATE TABLE analytics.dim_date AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d AS full_date,
  year(d) AS year,
  quarter(d) AS quarter,
  month(d) AS month,
  trunc(d, 'MM') AS month_start,
  date_sub(d, ((weekday(d)))) AS week_start
FROM (
  SELECT explode(sequence(
    to_date('2025-07-01'),
    to_date('2026-06-30'),
    interval 1 day
  )) AS d
);

In [0]:
create table analytics.dim_member as 
select row_number() over (order by member_id) as member_sk,
member_id,
gender,
city,
state
from raw.members


In [0]:
create table analytics.dim_hospital as 
select row_number() over (order by hospital_id) as hospital_sk,
hospital_id,
name as hospital_name,
city,
city_tier,
state
from raw.hospitals

In [0]:
create table analytics.dim_plan as 
select row_number() over (order by p.member_id, p.policy_id) as plan_sk,
p.member_id,
p.policy_id,
p.plan_code,
pl.plan_family,
pl.sum_insured,
pl.room_rent_cap_per_day,
pl.annual_premium,
p.effective_from,
p.effective_till,
p.is_current
from silver.dim_policy_scd2 p 
join raw.plans pl 
on p.plan_code = pl.plan_code

In [0]:
CREATE TABLE analytics.fact_claims AS
SELECT
  c.claim_id,
  m.member_sk,
  h.hospital_sk,
  pl.plan_sk,
  da.date_key AS admission_date_key,
  ds.date_key AS submitted_date_key,
  dse.date_key AS settled_date_key,
  c.claim_type,
  c.status,
  c.rejection_code,
  c.is_network_at_admission,
  c.room_rent_over_cap,
  CAST(c.billed_amount AS DECIMAL(12,2))   AS billed_amount,
  CAST(c.approved_amount AS DECIMAL(12,2)) AS approved_amount,
  datediff(c.discharge_date, c.admission_date) AS length_of_stay,
  datediff(to_date(c.settled_ts), to_date(c.submitted_ts)) AS turnaround_days
FROM claims.gold.claims_enriched c
LEFT JOIN claims.analytics.dim_member   m  ON c.member_id   = m.member_id
LEFT JOIN claims.analytics.dim_hospital h  ON c.hospital_id = h.hospital_id
LEFT JOIN claims.analytics.dim_plan     pl
  ON  c.member_id = pl.member_id
  AND c.policy_id = pl.policy_id
  AND c.admission_date >= pl.effective_from
  AND (c.admission_date <= pl.effective_till OR pl.effective_till IS NULL)
LEFT JOIN claims.analytics.dim_date da  ON c.admission_date = da.full_date
LEFT JOIN claims.analytics.dim_date ds  ON to_date(c.submitted_ts) = ds.full_date
LEFT JOIN claims.analytics.dim_date dse ON to_date(c.settled_ts) = dse.full_date;